# Treino do detector de EPI (capacete e óculos) — YOLO26n

Rode no Google Colab com GPU: **Ambiente de execução → Alterar o tipo de ambiente → GPU T4**.

Etapas: instalar dependências → baixar dataset do Roboflow → filtrar as 4 classes → treinar → avaliar → baixar `best.pt`.

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

## 1. Clonar o repositório (para usar o script de filtro)

In [ ]:
REPO = "https://github.com/emersonscc/detector-EPI.git"
!git clone -q {REPO}
%cd epi-detector

## 2. Baixar o dataset PPEs do Roboflow
A API key fica em roboflow.com → Settings → API Keys. Ela é pedida na hora e não fica salva no notebook.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("ramcomills").project("ppes-kaxsi-v4bep")
ds = project.version(1).download("yolov8", location="data/raw/ppes")

## 3. Filtrar para helmet / no_helmet / goggles / no_goggles

In [ ]:
!python scripts/filter_dataset.py --src data/raw/ppes --dst data/processed/epi

## 4. Treinar
`yolo26n.pt` é baixado automaticamente com pesos pré-treinados no COCO (transfer learning).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
model.train(
    data="data/processed/epi/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,      # para cedo se não melhorar
    project="runs",
    name="epi_yolo26n",
)

## 5. Avaliar no conjunto de teste

In [ ]:
best = YOLO("runs/epi_yolo26n/weights/best.pt")
metrics = best.val(data="data/processed/epi/data.yaml", split="test")
print(f"mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}")
for i, name in best.names.items():
    print(f"{name:>11}: mAP50 = {metrics.box.maps[i]:.3f}")

## 6. Ver algumas predições

In [ ]:
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob("data/processed/epi/test/images/*"))[:6]
results = best.predict(imgs, conf=0.4, save=True, project="runs", name="preview", exist_ok=True)
for p in sorted(glob.glob("runs/preview/*"))[:6]:
    display(Image(filename=p, width=500))

## 7. Baixar o modelo e os gráficos
Coloque o `best.pt` em `models/` no seu computador. Os gráficos (curvas, matriz de confusão) servem para o relatório.

In [ ]:
!cd runs/epi_yolo26n && zip -qr ../../resultados_treino.zip weights/best.pt *.png *.jpg results.csv
from google.colab import files
files.download("resultados_treino.zip")